# 🪙 ETL Crypto Pipeline

![Python](https://img.shields.io/badge/Python-3.10+-blue?logo=python&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-2.0+-150458?logo=pandas&logoColor=white)
![API](https://img.shields.io/badge/API-CoinGecko-green)
![Status](https://img.shields.io/badge/Status-Complete-brightgreen)

---

An end-to-end ETL pipeline that **extracts** cryptocurrency market data from the CoinGecko public API, **transforms** it into a clean structured format using pandas, and **loads** it into a CSV file ready for analysis.

Built as part of a Data Engineering portfolio project.

| Field | Details |
|---|---|
| **Data Source** | [CoinGecko API](https://www.coingecko.com/en/api) |
| **Output** | `data/crypto_data.csv` |
| **Environment** | Google Colab |


## 📋 Project Overview

This ETL pipeline performs the following steps:

| Step | Description | Module |
|------|-------------|--------|
| **Extract** | Fetch top 10 cryptocurrencies by market cap from CoinGecko | `extract.py` |
| **Transform** | Select relevant columns, add timestamp, validate data | `transform.py` |
| **Load** | Save processed data to CSV with error handling | `load.py` |

```
CoinGecko API  →  extract.py  →  transform.py  →  load.py  →  crypto_data.csv
```


## 🛠️ Tech Stack

| Tool | Purpose |
|---|---|
| **Python 3.10+** | Core language |
| **Pandas** | Data transformation and analysis |
| **Requests** | HTTP calls to CoinGecko API |
| **CSV** | Lightweight data storage |
| **Google Colab** | Cloud execution environment |


## ⚙️ Setup

Install required dependencies before running the pipeline.

In [1]:
# Install dependencies (only needed if not already installed)
!pip install requests pandas --quiet

---
## 🔵 Step 1 — Extract

Fetch the top 10 cryptocurrencies by market cap from the CoinGecko public API.

- **Endpoint**: `/coins/markets`
- **Currency**: USD
- **Records**: 10 coins ordered by market cap (descending)
- **Error handling**: Returns `None` on API failure with descriptive logging

In [2]:
%%writefile extract.py

import requests
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

API_URL = "https://api.coingecko.com/api/v3/coins/markets"

def extract_data(vs_currency: str = "usd", top_n: int = 10) -> list | None:
    """
    Fetch top N cryptocurrencies by market cap from CoinGecko API.

    Args:
        vs_currency: Target currency for prices (default: 'usd').
        top_n: Number of coins to retrieve (default: 10).

    Returns:
        List of coin data dicts, or None if the request fails.
    """
    params = {
        "vs_currency": vs_currency,
        "order": "market_cap_desc",
        "per_page": top_n,
        "page": 1,
    }

    try:
        response = requests.get(API_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        logger.info(f"Successfully extracted {len(data)} records from CoinGecko.")
        return data

    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP error: {e}")
    except requests.exceptions.ConnectionError:
        logger.error("Connection error. Check your internet connection.")
    except requests.exceptions.Timeout:
        logger.error("Request timed out after 10 seconds.")
    except Exception as e:
        logger.error(f"Unexpected error during extraction: {e}")

    return None

Writing extract.py


---
## 🟡 Step 2 — Transform

Convert the raw JSON response into a clean, structured pandas DataFrame.

- Selects relevant columns (`id`, `symbol`, `name`, `current_price`, `market_cap`, `total_volume`)
- Adds a `extracted_at` timestamp to track when the data was fetched
- Validates that no critical columns contain null values
- Normalizes symbol to uppercase

In [3]:
%%writefile transform.py

import pandas as pd
import logging
from datetime import datetime, timezone

logger = logging.getLogger(__name__)

COLUMNS = ["id", "symbol", "name", "current_price", "market_cap", "total_volume"]

def transform_data(data: list) -> pd.DataFrame | None:
    """
    Transform raw API response into a clean, structured DataFrame.

    Args:
        data: Raw list of coin dicts from the CoinGecko API.

    Returns:
        Cleaned pandas DataFrame, or None if transformation fails.
    """
    if not data:
        logger.warning("No data received for transformation.")
        return None

    try:
        df = pd.DataFrame(data)

        # Select and reorder relevant columns
        df = df[COLUMNS].copy()

        # Add extraction timestamp (UTC)
        df["extracted_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

        # Validate: drop rows with nulls in critical numeric columns
        critical_cols = ["current_price", "market_cap", "total_volume"]
        before = len(df)
        df.dropna(subset=critical_cols, inplace=True)
        dropped = before - len(df)
        if dropped > 0:
            logger.warning(f"Dropped {dropped} row(s) with missing critical values.")

        # Normalize symbol to uppercase
        df["symbol"] = df["symbol"].str.upper()

        logger.info(f"Transformation complete. {len(df)} records ready to load.")
        return df

    except KeyError as e:
        logger.error(f"Missing expected column in data: {e}")
    except Exception as e:
        logger.error(f"Unexpected error during transformation: {e}")

    return None

Writing transform.py


---
## 🟢 Step 3 — Load

Persist the transformed DataFrame to a CSV file.

- Creates the output directory automatically if it doesn't exist
- Saves to `data/crypto_data.csv`
- Returns a boolean to indicate success or failure

In [4]:
%%writefile load.py

import os
import pandas as pd
import logging

logger = logging.getLogger(__name__)

OUTPUT_PATH = "data/crypto_data.csv"

def load_data(df: pd.DataFrame) -> bool:
    """
    Save transformed DataFrame to a CSV file.

    Args:
        df: Cleaned DataFrame to persist.

    Returns:
        True if the file was saved successfully, False otherwise.
    """
    if df is None or df.empty:
        logger.warning("No data to load. Skipping file write.")
        return False

    try:
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        df.to_csv(OUTPUT_PATH, index=False)
        logger.info(f"{len(df)} records saved to '{OUTPUT_PATH}'.")
        return True

    except PermissionError:
        logger.error(f"Permission denied when writing to '{OUTPUT_PATH}'.")
    except Exception as e:
        logger.error(f"Unexpected error during load: {e}")

    return False

Writing load.py


---
## 🚀 Run ETL Pipeline

Execute the full pipeline end-to-end by running `main.py`.

In [5]:
%%writefile main.py

import logging
from extract import extract_data
from transform import transform_data
from load import load_data

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)


def run_pipeline():
    """Orchestrate the full Extract -> Transform -> Load pipeline."""
    logger.info("=" * 40)
    logger.info("Starting ETL Crypto Pipeline...")
    logger.info("=" * 40)

    # Step 1: Extract
    raw_data = extract_data()
    if raw_data is None:
        logger.error("Pipeline aborted: extraction failed.")
        return

    logger.info(f"Extracted {len(raw_data)} raw records.")

    # Step 2: Transform
    df = transform_data(raw_data)
    if df is None:
        logger.error("Pipeline aborted: transformation failed.")
        return

    # Step 3: Load
    success = load_data(df)

    if success:
        logger.info("Pipeline completed successfully.")
    else:
        logger.error("Pipeline finished with errors during load.")


if __name__ == "__main__":
    run_pipeline()

Writing main.py


In [6]:
# Run the full ETL pipeline
!python main.py

INFO: NumExpr defaulting to 2 threads.
INFO: ========================================
INFO: Starting ETL Crypto Pipeline...
INFO: ========================================
INFO: Successfully extracted 10 records from CoinGecko.
INFO: Extracted 10 raw records.
INFO: Transformation complete. 10 records ready to load.
INFO: 10 records saved to 'data/crypto_data.csv'.
INFO: Pipeline completed successfully.


---
## 📊 Output Preview

Verify the output file and inspect the resulting dataset.

In [7]:
import pandas as pd

df = pd.read_csv("data/crypto_data.csv")

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Extracted at: {df['extracted_at'].iloc[0]}")
print()
df.head(10)

Shape: 10 rows x 7 columns
Extracted at: 2026-03-23 14:08:09 UTC



,id,symbol,name,current_price,market_cap,total_volume,extracted_at
0,bitcoin,BTC,Bitcoin,71443.000000,1431380313428,48556023292,2026-03-23 14:08:09 UTC
1,ethereum,ETH,Ethereum,2182.730000,263787273755,24931906313,2026-03-23 14:08:09 UTC
2,tether,USDT,Tether,0.999763,184141318375,79941610397,2026-03-23 14:08:09 UTC
3,ripple,XRP,XRP,1.440000,88557892496,3048561191,2026-03-23 14:08:09 UTC
4,binancecoin,BNB,BNB,646.940000,88331242895,1436183161,2026-03-23 14:08:09 UTC
5,usd-coin,USDC,USDC,1.000000,78849931717,8156255233,2026-03-23 14:08:09 UTC
6,solana,SOL,Solana,91.320000,52249240788,4353193585,2026-03-23 14:08:09 UTC
7,tron,TRX,TRON,0.310409,29421388339,743903255,2026-03-23 14:08:09 UTC
8,figure-heloc,FIGR_HELOC,Figure Heloc,1.019000,16139424812,107369,2026-03-23 14:08:09 UTC
9,dogecoin,DOGE,Dogecoin,0.094209,14469122398,1364316743,2026-03-23 14:08:09 UTC


In [8]:
# Quick summary statistics on numeric columns
df[["current_price", "market_cap", "total_volume"]].describe()

,current_price,market_cap,total_volume
count,10.000000,1.000000e+01,1.000000e+01
mean,7436.885338,2.247327e+11,1.725321e+10
std,22499.927901,4.311768e+11,2.685861e+10
min,0.094209,1.446912e+10,1.073690e+05
25%,0.999822,3.512835e+10,1.382283e+09
50%,1.229500,8.359059e+10,3.700877e+09
75%,508.035000,1.602455e+11,2.073799e+10
max,71443.000000,1.431380e+12,7.994161e+10


---
## ✅ Conclusion

This project demonstrates a **production-style ETL pipeline** with:

| Concept | Implementation |
|---|---|
| API data extraction | CoinGecko REST API with error handling |
| Data transformation | pandas — column selection, normalization, timestamps |
| Data validation | Null checks and row dropping with logging |
| Modular design | Separate `extract`, `transform`, `load` modules |
| Logging | Python `logging` module throughout all steps |
| Data persistence | CSV output with directory auto-creation |

### 🔭 Potential Extensions

- **Database storage** → PostgreSQL or BigQuery instead of CSV
- **Scheduling** → Apache Airflow or Prefect DAGs
- **Cloud deployment** → AWS Lambda / GCP Cloud Functions
- **Data quality checks** → Great Expectations or Pandera
- **Dashboard** → Connect output to Metabase or Looker Studio
